In [ ]:
from __future__ import annotations

import json
import time
import logging
from datetime import datetime
from typing import Any

logger = logging.getLogger(__name__)


def update_pipeline_progress(
    *,
    project_doc: dict,
    projects_container,
    stages: list[str],
    current_stage: str,
    error: str | None = None,
    extra: dict[str, Any] | None = None,
    last_updated: datetime | None = None,
) -> dict:
    """
    Generic pipeline progress tracker.

    Features
    --------
    1. Tracks:
       - current stage
       - last completed stage
       - overall pipeline status
       - percentage progress
       - timestamps
       - errors

    2. Status rules:
       - First stage      -> "started"
       - Middle stages    -> "running"
       - Final stage      -> "completed"
       - Any error        -> "error"

    3. Agnostic design:
       - Pass ordered list of stages
       - Works for any pipeline

    Parameters
    ----------
    project_doc : dict
        Existing project document.

    projects_container :
        Cosmos DB / database container supporting upsert_item().

    stages : list[str]
        Ordered pipeline stages.

    current_stage : str
        Current active stage.

    error : str | None
        Error message if pipeline failed.

    extra : dict | None
        Optional metadata.

    last_updated : datetime | None
        Optional timestamp override.

    Returns
    -------
    dict
        Generated progress payload.
    """

    if current_stage not in stages:
        raise ValueError(f"Invalid stage '{current_stage}'. Allowed stages: {stages}")

    total_stages = len(stages)
    current_index = stages.index(current_stage)

    # ---------------------------------------------------------
    # Progress %
    # ---------------------------------------------------------
    percentage = round(((current_index + 1) / total_stages) * 100, 2)

    # ---------------------------------------------------------
    # Status
    # ---------------------------------------------------------
    if error:
        status = "error"
    elif current_index == 0:
        status = "started"
    elif current_index == total_stages - 1:
        status = "completed"
    else:
        status = "running"

    # ---------------------------------------------------------
    # Last completed stage
    # ---------------------------------------------------------
    last_completed_stage = stages[current_index - 1] if current_index > 0 else None
    logger.info(f"Last Completed Stage: {last_completed_stage}")

    # ---------------------------------------------------------
    # Build payload
    # ---------------------------------------------------------
    payload = {
        "pipeline_status": status,
        "current_stage": current_stage,
        "last_completed_stage": last_completed_stage,
        "current_step": current_index + 1,
        "total_steps": total_stages,
        "percentage": percentage,
        "error": error,
        "last_updated": (last_updated or datetime.utcnow()).isoformat(),
        "extra": extra or {},
        "ts": time.time(),
    }

    # ---------------------------------------------------------
    # Save in project doc
    # ---------------------------------------------------------
    project_doc["Pipeline_Progress"] = payload

    projects_container.upsert_item(project_doc)

    # ---------------------------------------------------------
    # Console logging
    # ---------------------------------------------------------
    logger.info(
        json.dumps(
            {
                "type": "pipeline_progress",
                **payload,
            },
            indent=2,
        ),
        flush=True,
    )

    return payload

In [ ]:
# =========================================================
# MOCK COSMOS CONTAINER
# =========================================================
class MockCosmosContainer:
    """
    Mock version of Cosmos DB container for testing.
    """

    def upsert_item(self, item: dict):
        print("\n================ UPSERT TO COSMOS ================\n")
        print(json.dumps(item, indent=2))
        print("\n==================================================\n")



# =========================================================
# MOCK TEST
# =========================================================
if __name__ == "__main__":

    # -----------------------------------------------------
    # Mock project document
    # -----------------------------------------------------
    project_doc = {
        "id": "project_001",
        "project_name": "TRF Generation",
    }

    # -----------------------------------------------------
    # Mock Cosmos container
    # -----------------------------------------------------
    projects_container = MockCosmosContainer()

    # -----------------------------------------------------
    # Ordered stages
    # -----------------------------------------------------
    STAGES = [
        "download_files",
        "extract_text",
        "build_vectorstore",
        "generate_trf",
        "upload_outputs",
    ]

    # -----------------------------------------------------
    # Simulate full pipeline
    # -----------------------------------------------------
    for stage in STAGES:

        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=stage,
            extra={
                "info": f"{stage} completed successfully"
            },
        )

        time.sleep(1)

    # -----------------------------------------------------
    # Simulate error
    # -----------------------------------------------------
    update_pipeline_progress(
        project_doc=project_doc,
        projects_container=projects_container,
        stages=STAGES,
        current_stage="generate_trf",
        error="OpenAI request timeout",
    )

C:\Users\saurav\AppData\Local\Temp\ipykernel_21252\3841737922.py:114: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": (last_updated or datetime.utcnow()).isoformat(),



================ UPSERT TO COSMOS ================

{
  "id": "project_001",
  "project_name": "TRF Generation",
  "Pipeline_Progress": {
    "pipeline_status": "started",
    "current_stage": "download_files",
    "last_completed_stage": null,
    "current_step": 1,
    "total_steps": 5,
    "percentage": 20.0,
    "error": null,
    "last_updated": "2026-05-14T07:27:01.143172",
    "extra": {
      "info": "download_files completed successfully"
    },
    "ts": 1778743621.1431875
  }
}



================ UPSERT TO COSMOS ================

{
  "id": "project_001",
  "project_name": "TRF Generation",
  "Pipeline_Progress": {
    "pipeline_status": "running",
    "current_stage": "extract_text",
    "last_completed_stage": "download_files",
    "current_step": 2,
    "total_steps": 5,
    "percentage": 40.0,
    "error": null,
    "last_updated": "2026-05-14T07:27:02.144081",
    "extra": {
      "info": "extract_text completed successfully"
    },
    "ts": 1778743622.1440902
  }
}


In [ ]:
def download_files():
    print("files downloaded")

def extract_text():
    print("Text extracted")

def build_vectorstore():
    print("build vectorstore")

def generate_trf():
    print("trf generated")

def upload_outputs():
    print("outputs uploaded")

STAGES = [
    "download_files",
    "extract_text",
    "build_vectorstore",
    "generate_trf",
    "upload_outputs",
]


def main(
    project_doc: dict,
    projects_container,
):
    """
    Main pipeline runner.
    """

    try:
        current_stage = None

        # =====================================================
        # STAGE 1
        # =====================================================
        current_stage = "download_files"
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            extra={
                "message": "Downloading input files"
            },
        )

        download_files()

        # =====================================================
        # STAGE 2
        # =====================================================
        current_stage="extract_text"
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            extra={
                "message": "Extracting PDF text"
            },
        )

        extract_text()

        # =====================================================
        # STAGE 3
        # =====================================================
        current_stage="build_vectorstore"
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            extra={
                "message": "Building vector database"
            },
        )

        build_vectorstore()

        # =====================================================
        # STAGE 4
        # =====================================================
        current_stage="generate_trf"
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            extra={
                "message": "Generating TRF"
            },
        )

        generate_trf()

        # =====================================================
        # FINAL STAGE
        # =====================================================
        current_stage="upload_outputs"
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            extra={
                "message": "Uploading final outputs"
            },
        )

        upload_outputs()

        print("\n PIPELINE COMPLETED\n")

    except Exception as e:

        # -----------------------------------------------------
        # Mark pipeline failed
        # -----------------------------------------------------
        update_pipeline_progress(
            project_doc=project_doc,
            projects_container=projects_container,
            stages=STAGES,
            current_stage=current_stage,
            error=str(e),
            extra={
                "message": "Pipeline execution failed"
            },
        )

        raise

In [ ]:
project_doc = {
        "id": "project_001",
        "project_name": "TRF Pipeline",
    }

project_container = MockCosmosContainer()

main(
    project_doc=project_doc,
    projects_container=project_container,
)


================ UPSERT TO COSMOS ================

{
  "id": "project_001",
  "project_name": "TRF Pipeline",
  "Pipeline_Progress": {
    "pipeline_status": "started",
    "current_stage": "download_files",
    "last_completed_stage": null,
    "current_step": 1,
    "total_steps": 5,
    "percentage": 20.0,
    "error": null,
    "last_updated": "2026-05-14T06:08:21.356734",
    "extra": {
      "message": "Downloading input files"
    },
    "ts": 1778738901.3567455
  }
}


files downloaded

================ UPSERT TO COSMOS ================

{
  "id": "project_001",
  "project_name": "TRF Pipeline",
  "Pipeline_Progress": {
    "pipeline_status": "running",
    "current_stage": "extract_text",
    "last_completed_stage": "download_files",
    "current_step": 2,
    "total_steps": 5,
    "percentage": 40.0,
    "error": null,
    "last_updated": "2026-05-14T06:08:21.356933",
    "extra": {
      "message": "Extracting PDF text"
    },
    "ts": 1778738901.3569376
  }
}


Text extr

C:\Users\saurav\AppData\Local\Temp\ipykernel_21252\2532558028.py:113: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": (last_updated or datetime.utcnow()).isoformat(),
